# LungCT 4DCT Mask Inspection

Run this notebook inside `py310sitreg_v1_test1` from `/homebase/code_sync/inrorspline`.

This notebook stages paired `T00` and `T50` lung masks with `lungmask R231`, verifies fixed `T00` compatibility, and renders coronal QA overlays.

In [ ]:
from pathlib import Path

REPO = Path('/homebase/code_sync/inrorspline')
DATA_ROOT = Path('/database/datasets/lungCT/DIR-Lab')
STAGE_ROOT = DATA_ROOT / 'processed/lungmask_R231_native'
PREVIOUS_ROOT = Path('/database/wavereg/task4_lungct/masks')
OUT_DIR = REPO / 'outputs/data_inspection/lungct_4dct_mask_audit_r231_t00_t50'
REPO.exists(), DATA_ROOT.exists(), PREVIOUS_ROOT.exists()

## Generate staged masks

This can take time and requires the `lungmask` package/model in the container. Use `--overwrite` only when intentionally replacing an existing staging run.

In [ ]:
!cd /homebase/code_sync/inrorspline && if test -f /database/datasets/lungCT/DIR-Lab/processed/lungmask_R231_native/4DCT/generation_summary.json; then \
  echo 'Staged masks already exist; skipping generation.'; \
else \
  python scripts/data/preprocessing/lungct/generate_4dct_lung_masks.py \
  --case-ids 1-10 \
  --phases 00 50 \
  --data-root /database/datasets/lungCT/DIR-Lab \
  --stage-root /database/datasets/lungCT/DIR-Lab/processed/lungmask_R231_native \
  --modelname R231 \
  --device cuda; \
fi

## Verify fixed masks

This command is verify-only by default. Add `--execute` only after reviewing the report and intentionally promoting the staged masks.

In [ ]:
!cd /homebase/code_sync/inrorspline && python scripts/data/preprocessing/lungct/promote_4dct_lung_masks.py \
  --stage-root /database/datasets/lungCT/DIR-Lab/processed/lungmask_R231_native \
  --processed-root /database/datasets/lungCT/DIR-Lab/processed/lungmask_R231_native \
  --compat-root /database/wavereg/task4_lungct/masks \
  --require-fixed-match

In [ ]:
!cd /homebase/code_sync/inrorspline && python scripts/data/inspection/lungct_mask_visual_audit.py \
  --data-root /database/datasets/lungCT/DIR-Lab \
  --mask-root /database/datasets/lungCT/DIR-Lab/processed/lungmask_R231_native \
  --previous-mask-root /database/wavereg/task4_lungct/masks \
  --out-dir outputs/data_inspection/lungct_4dct_mask_audit_r231_t00_t50

In [ ]:
import pandas as pd

summary_csv = OUT_DIR / 'lungct_4dct_mask_visual_audit_summary.csv'
lung_df = pd.read_csv(summary_csv)
lung_df[['case_id', 'previous_fixed_exact_match', 'fixed_mask_fraction', 'moving_mask_fraction', 'figure']]

## Figure Review

Each panel shows fixed `T00`, generated fixed mask, old/new fixed-mask check, moving `T50`, generated moving mask, and moving-mask contour. Coronal panels use the review convention top = superior and bottom = inferior.

In [ ]:
from IPython.display import Image, Markdown, display

for row in lung_df.sort_values('case_id').itertuples(index=False):
    status = 'match' if bool(row.previous_fixed_exact_match) else 'DIFF'
    display(Markdown(f"### 4DCT case {int(row.case_id):02d} | previous T00 check: {status}"))
    fig_path = Path(row.figure)
    if not fig_path.is_absolute():
        fig_path = REPO / fig_path
    display(Image(filename=str(fig_path), width=1100))